In [1]:
# ============================================================
# Mini InstructGPT v2 — Phase 4: PPO Training
# Fix: optimizer states on CPU (saves ~1.4GB VRAM)
# GPT-2 Medium is too large for full GPU training on 4GB
# ============================================================

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW

# ── 1. Config ────────────────────────────────────────────────
CONFIG = {
    "sft_model_path":    "/kaggle/input/datasets/arnavx/mini-instructgpt-models/sft_model_v2/sft_model_v2",
    "reward_model_path": "/kaggle/input/datasets/arnavx/mini-instructgpt-models/reward_model_v2/reward_model_v2",
    "output_dir":        "/kaggle/working/ppo_model_v2",
    "max_prompt_len":    96,
    "max_new_tokens":    48,
    "batch_size":        1,
    "lr":                1e-6,
    "kl_coef":           0.7,
    "deflection_penalty_coef": 0.4,
    "num_steps":         120,
    "log_every":         10,
    "num_prompts":       500,
}

INAPPROPRIATE_PATTERNS = [
    r'\bfuck\b', r'\bshit\b', r'\bcunt\b', r'\bfucker\b',
    r'\bporn\b', r'\bnude\b', r'\bsuicide\b', r'\bcum\b',
    r'\bcocaine\b', r'\bheroin\b', r'\brake\b', r'\bdick\b',
    r'cuss word', r'swear word', r'\bwhore\b', r'\bslut\b',
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.cuda.empty_cache()


# ── 2. Load Actor (GPU) with gradient checkpointing ─────────
print("\nLoading actor on GPU...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["sft_model_path"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

actor = AutoModelForCausalLM.from_pretrained(CONFIG["sft_model_path"]).to(device)
actor.gradient_checkpointing_enable()  # trades compute for memory
actor.train()
print(f"Actor loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 3. Reference Model (CPU) ─────────────────────────────────
print("\nLoading reference model on CPU...")
reference = AutoModelForCausalLM.from_pretrained(CONFIG["sft_model_path"])
for param in reference.parameters():
    param.requires_grad = False
reference.eval()
reference = reference.to("cpu")
print(f"Reference on CPU. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 4. Reward Model (CPU) ────────────────────────────────────
print("\nLoading reward model v2 on CPU...")
reward_tokenizer = AutoTokenizer.from_pretrained(CONFIG["reward_model_path"])
reward_tokenizer.pad_token = reward_tokenizer.eos_token
reward_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["reward_model_path"])
reward_model.eval()
reward_model = reward_model.to("cpu")
print(f"All models loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 5. CPU Optimizer ─────────────────────────────────────────
# 🧠 KEY FIX: move model params to CPU for optimizer
#    AdamW stores exp_avg and exp_avg_sq (same size as params)
#    For 345M model that's ~1.4GB just for optimizer states
#    By keeping params on CPU during optimizer.step() we avoid OOM

class CPUOffloadOptimizer:
    """
    Optimizer that keeps optimizer states on CPU.
    During step: move grads to CPU → update → move params back to GPU
    This saves ~1.4GB VRAM for GPT-2 Medium.
    """
    def __init__(self, model, lr):
        self.model = model
        # Create CPU copies of params for optimizer
        self.cpu_params = [
            p.detach().cpu().float().requires_grad_(True)
            for p in model.parameters() if p.requires_grad
        ]
        self.optimizer = AdamW(self.cpu_params, lr=lr)

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step(self, gpu_params):
        # Copy gradients from GPU to CPU params
        for cpu_p, gpu_p in zip(self.cpu_params, gpu_params):
            if gpu_p.grad is not None:
                cpu_p.grad = gpu_p.grad.detach().cpu().float()

        # Clip gradients on CPU
        torch.nn.utils.clip_grad_norm_(self.cpu_params, 1.0)

        # Optimizer step on CPU
        self.optimizer.step()

        # Copy updated params back to GPU
        with torch.no_grad():
            for cpu_p, gpu_p in zip(self.cpu_params, gpu_params):
                gpu_p.copy_(cpu_p.to(device).to(gpu_p.dtype))

gpu_params = [p for p in actor.parameters() if p.requires_grad]
optimizer = CPUOffloadOptimizer(actor, lr=CONFIG["lr"])


# —— 6. Load Prompts (safe PPO prompt pool, no HH sampling) ——
from datasets import load_dataset
import random

print("\nLoading safe PPO prompts...")
d = load_dataset("databricks/databricks-dolly-15k", split="train")

SAFE_CATS = {"open_qa", "closed_qa", "information_extraction", "summarization"}

prompts = []
for x in d:
    if x["category"] in SAFE_CATS:
        q = x["instruction"].strip()
        if q:
            prompts.append(f"Human: {q}\n\nAssistant:")

random.shuffle(prompts)
prompts = prompts[:CONFIG["num_prompts"]]

if len(prompts) == 0:
    raise ValueError("No safe prompts loaded.")

print(f"Loaded {len(prompts)} safe prompts")


# ── 7. Helpers ───────────────────────────────────────────────
DEFLECTION_PATTERNS = [
    r"\bcan i ask\b",
    r"\bis there anything else\b",
    r"\bcan you clarify\b",
    r"\bwhat do you mean\b",
    r"\bi['’]?m not sure\b",
    r"\bit depends\b",
]

def deflection_penalty(response_text):
    t = response_text.lower()
    penalty = 0.0
    for p in DEFLECTION_PATTERNS:
        if re.search(p, t):
            penalty += 0.25
    if len(response_text.split()) < 8:
        penalty += 0.20
    return penalty

def get_reward_score(prompt_text, response_text):
    full_text = prompt_text + response_text
    tokens = reward_tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    with torch.no_grad():
        score = reward_model(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
        ).logits.squeeze().item()
    return score

def compute_kl(actor_logits, ref_logits, response_ids):
    actor_lp = F.log_softmax(actor_logits.cpu().float(), dim=-1)
    ref_lp   = F.log_softmax(ref_logits.float(), dim=-1)
    actor_tlp = actor_lp.gather(-1, response_ids.cpu().unsqueeze(-1)).squeeze(-1)
    ref_tlp   = ref_lp.gather(-1, response_ids.cpu().unsqueeze(-1)).squeeze(-1)
    return (actor_tlp - ref_tlp).mean()


# ── 8. PPO Loop ──────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Starting PPO v2 — {CONFIG['num_steps']} steps")
print(f"Actor GPU | Reference CPU | Optimizer CPU offloaded")
print(f"{'='*60}\n")

step_rewards = []
step_kls = []

for step in range(CONFIG["num_steps"]):
    prompt_text = prompts[step % len(prompts)]

    # Tokenize
    enc = tokenizer(
      prompt_text,
      return_tensors="pt",
      truncation=True,
      max_length=CONFIG["max_prompt_len"],
    )
    prompt_ids = enc["input_ids"].to(device)
    prompt_mask = enc["attention_mask"].to(device)
    prompt_len = prompt_ids.shape[1]

    # Generate
    with torch.no_grad():
        full_ids = actor.generate(
            input_ids=prompt_ids,
            attention_mask=prompt_mask,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.3,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = full_ids[:, prompt_len:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    if not response_text.strip() or response_ids.shape[1] == 0:
        continue

    # Score
    reward_score = get_reward_score(prompt_text, response_text)
    step_rewards.append(reward_score)

    # Reference forward on CPU
    full_attn = torch.ones_like(full_ids, device=full_ids.device)
    with torch.no_grad():
        ref_logits = reference(
            input_ids=full_ids[:, :-1].cpu(),
            attention_mask=full_attn[:, :-1].cpu(),
        ).logits[:, prompt_len - 1:-1, :]

    # Actor forward on GPU
    actor_out = actor(
        input_ids=full_ids[:, :-1],
        attention_mask=full_attn[:, :-1],
    )
    actor_logits = actor_out.logits[:, prompt_len - 1:-1, :]

    # KL on CPU
    kl = compute_kl(actor_logits, ref_logits, response_ids[:, :actor_logits.shape[1]])
    step_kls.append(kl.item())

    kl_raw = kl.item()
    kl_pen = max(0.0, kl_raw)   # prevent negative KL from boosting reward

    # Total reward
    total_reward = (
        reward_score
        - CONFIG["kl_coef"] * kl_pen
        - CONFIG["deflection_penalty_coef"] * deflection_penalty(response_text)
    )

    # Policy gradient loss
    log_probs = F.log_softmax(actor_logits, dim=-1)
    token_log_probs = log_probs.gather(
        -1, response_ids[:, :actor_logits.shape[1]].unsqueeze(-1)
    ).squeeze(-1).mean()

    loss = -total_reward * token_log_probs

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step(gpu_params)

    torch.cuda.empty_cache()

    if (step + 1) % CONFIG["log_every"] == 0:
        recent_reward = sum(step_rewards[-10:]) / len(step_rewards[-10:])
        recent_kl = sum(step_kls[-10:]) / len(step_kls[-10:])
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"Step {step+1:3d}/{CONFIG['num_steps']} | "
              f"Reward: {recent_reward:+.3f} | "
              f"KL: {recent_kl:.3f} | "
              f"VRAM: {vram:.2f}GB")

        if (step + 1) % 50 == 0:
            print(f"  Prompt:   {prompt_text}")
            print(f"  Response: {response_text[:150]}\n")


# ── 9. Save ──────────────────────────────────────────────────
actor.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"\n✅ PPO model v2 saved to: {CONFIG['output_dir']}")


# ── 10. Final Comparison ─────────────────────────────────────
print("\n" + "="*60)
print("FINAL COMPARISON — Base vs PPO v1 vs PPO v2")
print("="*60)

def generate(mdl, tok, prompt, max_new=60):
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.3,
            use_cache=False,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

torch.cuda.empty_cache()

base_tok = AutoTokenizer.from_pretrained("gpt2")
base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

v1_tok = AutoTokenizer.from_pretrained("/kaggle/input/datasets/arnavx/mini-instructgpt-models/ppo_model/ppo_model", local_files_only=True)
v1_tok.pad_token = v1_tok.eos_token
ppo_v1 = AutoModelForCausalLM.from_pretrained("/kaggle/input/datasets/arnavx/mini-instructgpt-models/ppo_model/ppo_model", local_files_only=True).to(device)

test_prompts = [
    "Human: What is the capital of France?\n\nAssistant:",
    "Human: How do I make a cup of tea?\n\nAssistant:",
    "Human: Can you explain what gravity is?\n\nAssistant:",
    "Human: What are some tips for studying better?\n\nAssistant:",
]

for prompt in test_prompts:
    print(f"\n📝 {prompt}")
    print(f"  🔴 Base GPT-2 : {generate(base_model, base_tok, prompt)[:150]}")
    print(f"  🟡 PPO v1     : {generate(ppo_v1, v1_tok, prompt)[:150]}")
    print(f"  🟢 PPO v2     : {generate(actor, tokenizer, prompt)[:150]}")

print("\n" + "="*60)
print("🎉 Mini InstructGPT v2 Complete!")
print("="*60)
print("\nWhat you built:")
print("  ✅ GPT-2 Medium SFT (345M, 20k samples)")
print("  ✅ Self-play data generation (2000 pairs)")
print("  ✅ Reward model v2 — 76.5% accuracy (was 63%)")
print("  ✅ PPO v2 with CPU-offloaded optimizer")
print("  ✅ Novel: self-play inspired by SPIN paper")

Using device: cuda

Loading actor on GPU...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Actor loaded. VRAM: 1.44GB

Loading reference model on CPU...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Reference on CPU. VRAM: 1.44GB

Loading reward model v2 on CPU...


...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


📝 Human: What is the capital of France?

Assistant:
  🔴 Base GPT-2 :  The Capital.

 (The first part was translated into French at a later date.)
  🟡 PPO v1     :   Is there anything else you can help with this question please let us know. Can anyone explain more about what we mean by that than just say somethin
  🟢 PPO v2     :  Are you asking about how to answer this question, or are there some other questions that I should be able’t ask for your help in answering?"

📝 Human: How do I make a cup of tea?

Assistant:
  🔴 Base GPT-2 :  In order to use every part, it is necessary that you have knowledge on making the coffee. It may be hard for some people and very easy at others but 
  🟡 PPO v1     :  What can we say about this question and how could someone help us understand it better than is currently happening in our society.  Is there any othe
  🟢 PPO v2     :  That’s an interesting question!  What are some common questions you might have about this topic that people don't want to 

In [8]:
from datasets import load_dataset
import random, json

SAFE_CATS = {"open_qa", "closed_qa", "information_extraction", "summarization"}
DEFLECT_TEMPLATES = [
    "Can you clarify what you mean?",
    "Are you asking generally, or in a specific context?",
    "I’m not sure what you mean; could you rephrase?",
    "It depends on many factors. What exactly do you want to know?",
    "Could you give more context before I answer?"
]

d = load_dataset("databricks/databricks-dolly-15k", split="train")
pairs = []

for x in d:
    prompt = x["instruction"].strip()
    good = x["response"].strip()
    if not prompt or not good or len(good.split()) < 8:
        continue

    bad = random.choice(DEFLECT_TEMPLATES)
    pairs.append({
        "chosen":   f"Human: {prompt}\n\nAssistant: {good}",
        "rejected": f"Human: {prompt}\n\nAssistant: {bad}"
    })

random.shuffle(pairs)
pairs = pairs[:1000]  # start with 1000

out_path = "/kaggle/working/hard_negatives.jsonl"
with open(out_path, "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("Saved:", out_path, "pairs:", len(pairs))

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Saved: /kaggle/working/hard_negatives.jsonl pairs: 1000


In [9]:
from datasets import load_dataset, concatenate_datasets

hh = load_dataset("Anthropic/hh-rlhf", split="train").shuffle(seed=42).select(range(6000))
hn = load_dataset("json", data_files="/kaggle/working/hard_negatives.jsonl", split="train")

train_mix = concatenate_datasets([hh, hn, hn]).shuffle(seed=42)  # 2x hard negatives
train_mix.to_json("/kaggle/working/reward_train_mix.jsonl")
print(train_mix)

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Creating json from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 8000
})


In [10]:
!ls -la /kaggle/working

total 13820
drwxr-xr-x 3 root root     4096 Apr  2 06:48 .
drwxr-xr-x 5 root root     4096 Apr  2 06:38 ..
-rw-r--r-- 1 root root   710522 Apr  2 06:48 hard_negatives.jsonl
-rw-r--r-- 1 root root 13410354 Apr  2 06:48 reward_train_mix.jsonl
-rw-r--r-- 1 root root     9302 Apr  2 06:43 selfplay_reward_model.py
drwxr-xr-x 2 root root     4096 Apr  2 06:39 .virtual_documents


In [11]:
!find /kaggle -name "selfplay_reward_model.py"

/kaggle/working/selfplay_reward_model.py


In [16]:
%%writefile /kaggle/working/selfplay_reward_model.py
# ============================================================
# Mini InstructGPT v2 — Self-Play Reward Model (v3)
# Fix: completely isolate CPU training from CUDA context
# Resumes from step 200 checkpoint automatically
# ============================================================

import os
# Completely hide CUDA from the training process
# This prevents AdamW from touching CUDA context during CPU training
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import re
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# ── 1. Config ────────────────────────────────────────────────
CONFIG = {
    "sft_model_path":        "/kaggle/input/datasets/arnavx/mini-instructgpt-models/sft_model_v2/sft_model_v2",
    "checkpoint_path":       "/kaggle/input/datasets/arnavx/mini-instructgpt-models/reward_model_v2/reward_model_v2",  # resume from here
    "output_dir":            "/kaggle/working/reward_model_v3",
    "max_length":            128,
    "batch_size":            2,
    "grad_accum":            8,
    "epochs":                3,
    "lr":                    1e-5,
    "warmup_steps":          50,
    "num_hh_samples":        5000,
    "num_eval":              200,
    "log_every":             20,
    "save_every":            200,
    "selfplay_cache":        "./selfplay_pairs.json",
    "resume_from_step":      0,   # skip first 200 steps already trained
}

INAPPROPRIATE_PATTERNS = [
    r'\bfuck\b', r'\bshit\b', r'\bcunt\b', r'\bfucker\b',
    r'\bporn\b', r'\bnude\b', r'\bsuicide\b', r'\bcum\b',
    r'\bcocaine\b', r'\bheroin\b', r'\brake\b', r'\bdick\b',
    r'cuss word', r'swear word', r'\bwhore\b', r'\bslut\b',
]

# Force CPU — GPU completely hidden
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


# ── 2. Load Reward Model from Checkpoint ─────────────────────
# Resume from step 200 checkpoint — don't retrain from scratch!
print(f"\nLoading reward model from checkpoint: {CONFIG['checkpoint_path']}")
reward_tokenizer = AutoTokenizer.from_pretrained(CONFIG["sft_model_path"])
reward_tokenizer.pad_token = reward_tokenizer.eos_token
reward_tokenizer.model_max_length = CONFIG["max_length"]

reward_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["checkpoint_path"],
    num_labels=1,
)
reward_model.config.pad_token_id = reward_tokenizer.eos_token_id
reward_model = reward_model.to(device)
print("Resumed from checkpoint. Previous training not wasted! ✅")


# ── 3. Load Self-Play + HH-RLHF Data ─────────────────────────
def is_clean(sample):
    full_text = (sample["chosen"] + " " + sample["rejected"]).lower()
    return not any(re.search(p, full_text) for p in INAPPROPRIATE_PATTERNS)

def extract_last_turn(text):
    parts = text.split("\n\nAssistant:")
    if len(parts) > 1:
        last_human = parts[-2].split("\n\nHuman:")[-1].strip()
        last_assistant = parts[-1].strip()
        return f"Human: {last_human}\n\nAssistant: {last_assistant}"
    return text

print("\nLoading mixed reward training pairs...")
dataset = load_dataset("json", data_files={"train": "/kaggle/working/reward_train_mix.jsonl"}, split="train")

all_pairs = [{"chosen": x["chosen"], "rejected": x["rejected"]} for x in dataset]

import random
random.seed(42)
random.shuffle(all_pairs)

print(f"Total: {len(all_pairs):,} pairs")


# ── 4. Tokenize ──────────────────────────────────────────────
def tokenize_pair(sample):
    chosen = reward_tokenizer(
        sample["chosen"], truncation=True,
        max_length=CONFIG["max_length"], padding="max_length", return_tensors="pt"
    )
    rejected = reward_tokenizer(
        sample["rejected"], truncation=True,
        max_length=CONFIG["max_length"], padding="max_length", return_tensors="pt"
    )
    return {
        "chosen_input_ids":        chosen["input_ids"].squeeze(),
        "chosen_attention_mask":   chosen["attention_mask"].squeeze(),
        "rejected_input_ids":      rejected["input_ids"].squeeze(),
        "rejected_attention_mask": rejected["attention_mask"].squeeze(),
    }

print("Tokenizing...")
train_pairs = all_pairs[:-CONFIG["num_eval"]]
eval_pairs  = all_pairs[-CONFIG["num_eval"]:]

tok_train = Dataset.from_list(train_pairs).map(tokenize_pair, remove_columns=["chosen","rejected"], batched=False)
tok_eval  = Dataset.from_list(eval_pairs).map(tokenize_pair,  remove_columns=["chosen","rejected"], batched=False)
tok_train.set_format(type="torch")
tok_eval.set_format(type="torch")

train_loader = DataLoader(tok_train, batch_size=CONFIG["batch_size"], shuffle=True,  pin_memory=False)
eval_loader  = DataLoader(tok_eval,  batch_size=CONFIG["batch_size"], shuffle=False, pin_memory=False)

total_steps = (len(train_loader) // CONFIG["grad_accum"]) * CONFIG["epochs"]
remaining_steps = total_steps - CONFIG["resume_from_step"]
print(f"Total steps: {total_steps} | Remaining: {remaining_steps}")


# ── 5. Optimizer ─────────────────────────────────────────────
optimizer = AdamW(reward_model.parameters(), lr=CONFIG["lr"])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=CONFIG["warmup_steps"],
    num_training_steps=total_steps
)

# Fast-forward scheduler to match checkpoint step
for _ in range(CONFIG["resume_from_step"]):
    scheduler.step()


# ── 6. Eval ──────────────────────────────────────────────────
def evaluate():
    reward_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in eval_loader:
            chosen_input_ids = batch["chosen_input_ids"].to(device)
            chosen_attention_mask = batch["chosen_attention_mask"].to(device)
            rejected_input_ids = batch["rejected_input_ids"].to(device)
            rejected_attention_mask = batch["rejected_attention_mask"].to(device)

            cs = reward_model(
                input_ids=chosen_input_ids,
                attention_mask=chosen_attention_mask
            ).logits
            rs = reward_model(
                input_ids=rejected_input_ids,
                attention_mask=rejected_attention_mask
            ).logits

            correct += (cs > rs).sum().item()
            total += cs.shape[0]
    reward_model.train()
    return correct / total


# ── 7. Training Loop ─────────────────────────────────────────
print(f"\nResuming training from step {CONFIG['resume_from_step']}...")
print(f"Training on {device}.")

reward_model.train()
global_step = CONFIG["resume_from_step"]
accum_loss = 0.0
optimizer.zero_grad()
steps_done_this_run = 0

for epoch in range(CONFIG["epochs"]):
    print(f"\n--- Epoch {epoch+1}/{CONFIG['epochs']} ---")
    for step, batch in enumerate(train_loader):

        # Skip steps already trained in previous run
        if global_step - CONFIG["resume_from_step"] < 0:
            global_step += 1
            continue

        chosen_input_ids = batch["chosen_input_ids"].to(device)
        chosen_attention_mask = batch["chosen_attention_mask"].to(device)
        rejected_input_ids = batch["rejected_input_ids"].to(device)
        rejected_attention_mask = batch["rejected_attention_mask"].to(device)

        cs = reward_model(
            input_ids=chosen_input_ids,
            attention_mask=chosen_attention_mask
        ).logits
        rs = reward_model(
            input_ids=rejected_input_ids,
            attention_mask=rejected_attention_mask
        ).logits

        loss = -torch.nn.functional.logsigmoid(
            cs - rs
        ).mean() / CONFIG["grad_accum"]

        loss.backward()
        accum_loss += loss.item()

        if (step + 1) % CONFIG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            steps_done_this_run += 1

            if global_step % CONFIG["log_every"] == 0:
                print(f"Step {global_step}/{total_steps} | Loss: {accum_loss/CONFIG['log_every']:.4f}")
                accum_loss = 0.0

            if global_step % CONFIG["save_every"] == 0:
                reward_model.save_pretrained(f"{CONFIG['output_dir']}_step{global_step}")
                print(f"💾 Checkpoint saved at step {global_step}")

    acc = evaluate()
    print(f"✅ Epoch {epoch+1} accuracy: {acc*100:.1f}% (v1 was 63%)")


# ── 8. Save ──────────────────────────────────────────────────
reward_model.save_pretrained(CONFIG["output_dir"])
reward_tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"\n✅ Reward model v2 saved to: {CONFIG['output_dir']}")

final_acc = evaluate()
print(f"Final accuracy: {final_acc*100:.1f}%")
if final_acc > 0.63:
    print("🎉 Better than v1! Self-play helped.")
elif final_acc >= 0.58:
    print("⚠️  Similar to v1 — still good enough for PPO.")
else:
    print("❌ Needs more data — pipeline still works though.")

print("\nNext: python phase4_ppo_v2.py")

Overwriting /kaggle/working/selfplay_reward_model.py


In [17]:
!wc -l /kaggle/working/selfplay_reward_model.py

240 /kaggle/working/selfplay_reward_model.py


In [18]:
!head -n 20 /kaggle/working/selfplay_reward_model.py

# ============================================================
# Mini InstructGPT v2 — Self-Play Reward Model (v3)
# Fix: completely isolate CPU training from CUDA context
# Resumes from step 200 checkpoint automatically
# ============================================================

import os
# Completely hide CUDA from the training process
# This prevents AdamW from touching CUDA context during CPU training
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import re
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup



In [19]:
!python /kaggle/working/selfplay_reward_model.py

Using device: cuda

Loading reward model from checkpoint: /kaggle/input/datasets/arnavx/mini-instructgpt-models/reward_model_v2/reward_model_v2
Loading weights: 100%|█| 293/293 [00:00<00:00, 1431.00it/s, Materializing param=
Resumed from checkpoint. Previous training not wasted! ✅

Loading mixed reward training pairs...
Total: 8,000 pairs
Tokenizing...
Map: 100%|████████████████████████████| 200/200 [00:00<00:00, 497.85 examples/s]
Total steps: 1461 | Remaining: 1461

Resuming training from step 0...
Training on cuda.

--- Epoch 1/3 ---
Step 20/1461 | Loss: 1.2086
Step 40/1461 | Loss: 0.7860
Step 60/1461 | Loss: 0.6753
Step 80/1461 | Loss: 0.5959
Step 100/1461 | Loss: 0.5367
Step 120/1461 | Loss: 0.6030
Step 140/1461 | Loss: 0.5796
Step 160/1461 | Loss: 0.5512
Step 180/1461 | Loss: 0.5222
Step 200/1461 | Loss: 0.6035
Writing model shards: 100%|███████████████████████| 1/1 [00:02<00:00,  2.60s/it]
💾 Checkpoint saved at step 200
Step 220/1461 | Loss: 0.5200
Step 240/1461 | Loss: 0.5249
S

In [ ]:
from datasets import load_dataset
import re, json, random

INAPPROPRIATE_PATTERNS = [
    r'\bfuck\b', r'\bshit\b', r'\bcunt\b', r'\bfucker\b',
    r'\bporn\b', r'\bnude\b', r'\bsuicide\b', r'\bcum\b',
    r'\bcocaine\b', r'\bheroin\b', r'\brake\b', r'\bdick\b',
    r'cuss word', r'swear word', r'\bwhore\b', r'\bslut\b',
]

def is_clean(sample):
    t = (sample["chosen"] + " " + sample["rejected"]).lower()
    return not any(re.search(p, t) for p in INAPPROPRIATE_PATTERNS)

def extract_last_turn(text):
    parts = text.split("\n\nAssistant:")
    if len(parts) > 1:
        last_human = parts[-2].split("\n\nHuman:")[-1].strip()
        last_assistant = parts[-1].strip()
        return f"Human: {last_human}\n\nAssistant: {last_assistant}"
    return text

hh = load_dataset("Anthropic/hh-rlhf", split="train")
hh = hh.filter(is_clean).shuffle(seed=42).select(range(3000))

hh_pairs = []
for x in hh:
    hh_pairs.append({
        "chosen": extract_last_turn(x["chosen"]),
        "rejected": extract_last_turn(x["rejected"]),
    })

with open("/kaggle/working/hh_lastturn_3000.jsonl", "w", encoding="utf-8") as f:
    for p in hh_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print("saved hh_lastturn_3000:", len(hh_pairs))

In [1]:
!ls -lah /kaggle/input/datasets/arnavx/mini-instructgpt-models

total 4.0K
drwxr-xr-x 5 nobody nogroup    0 Apr  1 07:41 .
drwxr-xr-x 3 root   root    4.0K Apr  2 09:46 ..
drwxr-xr-x 3 nobody nogroup    0 Apr  1 07:41 ppo_model
drwxr-xr-x 3 nobody nogroup    0 Apr  1 07:41 reward_model_v2
drwxr-xr-x 3 nobody nogroup    0 Apr  1 07:41 sft_model_v2


In [2]:
!ls -lah /kaggle/working

total 12K
drwxr-xr-x 3 root root 4.0K Apr  2 09:46 .
drwxr-xr-x 5 root root 4.0K Apr  2 09:46 ..
drwxr-xr-x 2 root root 4.0K Apr  2 09:46 .virtual_documents


In [3]:
%%writefile /kaggle/working/phase4_ppo_v2.py
# ============================================================
# Mini InstructGPT v2 — Phase 4: PPO Training
# Fix: optimizer states on CPU (saves ~1.4GB VRAM)
# GPT-2 Medium is too large for full GPU training on 4GB
# ============================================================

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW

# ── 1. Config ────────────────────────────────────────────────
CONFIG = {
    "sft_model_path":    "/kaggle/input/datasets/arnavx/mini-instructgpt-models/sft_model_v2/sft_model_v2",
    "reward_model_path": "/kaggle/input/datasets/arnavx/mini-instructgpt-models/reward_model_v2/reward_model_v2",
    "output_dir":        "/kaggle/working/ppo_model_v2",
    "max_prompt_len":    96,
    "max_new_tokens":    48,
    "batch_size":        1,
    "lr":                1e-6,
    "kl_coef":           0.7,
    "deflection_penalty_coef": 0.4,
    "num_steps":         120,
    "log_every":         10,
    "num_prompts":       500,
}

INAPPROPRIATE_PATTERNS = [
    r'\bfuck\b', r'\bshit\b', r'\bcunt\b', r'\bfucker\b',
    r'\bporn\b', r'\bnude\b', r'\bsuicide\b', r'\bcum\b',
    r'\bcocaine\b', r'\bheroin\b', r'\brake\b', r'\bdick\b',
    r'cuss word', r'swear word', r'\bwhore\b', r'\bslut\b',
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.cuda.empty_cache()


# ── 2. Load Actor (GPU) with gradient checkpointing ─────────
print("\nLoading actor on GPU...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["sft_model_path"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

actor = AutoModelForCausalLM.from_pretrained(CONFIG["sft_model_path"]).to(device)
actor.gradient_checkpointing_enable()  # trades compute for memory
actor.train()
print(f"Actor loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 3. Reference Model (CPU) ─────────────────────────────────
print("\nLoading reference model on CPU...")
reference = AutoModelForCausalLM.from_pretrained(CONFIG["sft_model_path"])
for param in reference.parameters():
    param.requires_grad = False
reference.eval()
reference = reference.to("cpu")
print(f"Reference on CPU. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 4. Reward Model (CPU) ────────────────────────────────────
print("\nLoading reward model v2 on CPU...")
reward_tokenizer = AutoTokenizer.from_pretrained(CONFIG["reward_model_path"])
reward_tokenizer.pad_token = reward_tokenizer.eos_token
reward_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["reward_model_path"])
reward_model.eval()
reward_model = reward_model.to("cpu")
print(f"All models loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")


# ── 5. CPU Optimizer ─────────────────────────────────────────
# 🧠 KEY FIX: move model params to CPU for optimizer
#    AdamW stores exp_avg and exp_avg_sq (same size as params)
#    For 345M model that's ~1.4GB just for optimizer states
#    By keeping params on CPU during optimizer.step() we avoid OOM

class CPUOffloadOptimizer:
    """
    Optimizer that keeps optimizer states on CPU.
    During step: move grads to CPU → update → move params back to GPU
    This saves ~1.4GB VRAM for GPT-2 Medium.
    """
    def __init__(self, model, lr):
        self.model = model
        # Create CPU copies of params for optimizer
        self.cpu_params = [
            p.detach().cpu().float().requires_grad_(True)
            for p in model.parameters() if p.requires_grad
        ]
        self.optimizer = AdamW(self.cpu_params, lr=lr)

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step(self, gpu_params):
        # Copy gradients from GPU to CPU params
        for cpu_p, gpu_p in zip(self.cpu_params, gpu_params):
            if gpu_p.grad is not None:
                cpu_p.grad = gpu_p.grad.detach().cpu().float()

        # Clip gradients on CPU
        torch.nn.utils.clip_grad_norm_(self.cpu_params, 1.0)

        # Optimizer step on CPU
        self.optimizer.step()

        # Copy updated params back to GPU
        with torch.no_grad():
            for cpu_p, gpu_p in zip(self.cpu_params, gpu_params):
                gpu_p.copy_(cpu_p.to(device).to(gpu_p.dtype))

gpu_params = [p for p in actor.parameters() if p.requires_grad]
optimizer = CPUOffloadOptimizer(actor, lr=CONFIG["lr"])


# —— 6. Load Prompts (safe PPO prompt pool, no HH sampling) ——
from datasets import load_dataset
import random

print("\nLoading safe PPO prompts...")
d = load_dataset("databricks/databricks-dolly-15k", split="train")

SAFE_CATS = {"open_qa", "closed_qa", "information_extraction", "summarization"}

prompts = []
for x in d:
    if x["category"] in SAFE_CATS:
        q = x["instruction"].strip()
        if q:
            prompts.append(f"Human: {q}\n\nAssistant:")

random.shuffle(prompts)
prompts = prompts[:CONFIG["num_prompts"]]

if len(prompts) == 0:
    raise ValueError("No safe prompts loaded.")

print(f"Loaded {len(prompts)} safe prompts")


# ── 7. Helpers ───────────────────────────────────────────────
DEFLECTION_PATTERNS = [
    r"\bcan i ask\b",
    r"\bis there anything else\b",
    r"\bcan you clarify\b",
    r"\bwhat do you mean\b",
    r"\bi['’]?m not sure\b",
    r"\bit depends\b",
]

def deflection_penalty(response_text):
    t = response_text.lower()
    penalty = 0.0
    for p in DEFLECTION_PATTERNS:
        if re.search(p, t):
            penalty += 0.25
    if len(response_text.split()) < 8:
        penalty += 0.20
    return penalty

def get_reward_score(prompt_text, response_text):
    full_text = prompt_text + response_text
    tokens = reward_tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    with torch.no_grad():
        score = reward_model(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
        ).logits.squeeze().item()
    return score

def compute_kl(actor_logits, ref_logits, response_ids):
    actor_lp = F.log_softmax(actor_logits.cpu().float(), dim=-1)
    ref_lp   = F.log_softmax(ref_logits.float(), dim=-1)
    actor_tlp = actor_lp.gather(-1, response_ids.cpu().unsqueeze(-1)).squeeze(-1)
    ref_tlp   = ref_lp.gather(-1, response_ids.cpu().unsqueeze(-1)).squeeze(-1)
    return (actor_tlp - ref_tlp).mean()


# ── 8. PPO Loop ──────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"Starting PPO v2 — {CONFIG['num_steps']} steps")
print(f"Actor GPU | Reference CPU | Optimizer CPU offloaded")
print(f"{'='*60}\n")

step_rewards = []
step_kls = []

for step in range(CONFIG["num_steps"]):
    prompt_text = prompts[step % len(prompts)]

    # Tokenize
    enc = tokenizer(
      prompt_text,
      return_tensors="pt",
      truncation=True,
      max_length=CONFIG["max_prompt_len"],
    )
    prompt_ids = enc["input_ids"].to(device)
    prompt_mask = enc["attention_mask"].to(device)
    prompt_len = prompt_ids.shape[1]

    # Generate
    with torch.no_grad():
        full_ids = actor.generate(
            input_ids=prompt_ids,
            attention_mask=prompt_mask,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.3,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = full_ids[:, prompt_len:]
    response_text = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    if not response_text.strip() or response_ids.shape[1] == 0:
        continue

    # Score
    reward_score = get_reward_score(prompt_text, response_text)
    step_rewards.append(reward_score)

    # Reference forward on CPU
    full_attn = torch.ones_like(full_ids, device=full_ids.device)
    with torch.no_grad():
        ref_logits = reference(
            input_ids=full_ids[:, :-1].cpu(),
            attention_mask=full_attn[:, :-1].cpu(),
        ).logits[:, prompt_len - 1:-1, :]

    # Actor forward on GPU
    actor_out = actor(
        input_ids=full_ids[:, :-1],
        attention_mask=full_attn[:, :-1],
    )
    actor_logits = actor_out.logits[:, prompt_len - 1:-1, :]

    # KL on CPU
    kl = compute_kl(actor_logits, ref_logits, response_ids[:, :actor_logits.shape[1]])
    step_kls.append(kl.item())

    kl_raw = kl.item()
    kl_pen = max(0.0, kl_raw)   # prevent negative KL from boosting reward

    # Total reward
    total_reward = (
        reward_score
        - CONFIG["kl_coef"] * kl_pen
        - CONFIG["deflection_penalty_coef"] * deflection_penalty(response_text)
    )

    # Policy gradient loss
    log_probs = F.log_softmax(actor_logits, dim=-1)
    token_log_probs = log_probs.gather(
        -1, response_ids[:, :actor_logits.shape[1]].unsqueeze(-1)
    ).squeeze(-1).mean()

    loss = -total_reward * token_log_probs

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step(gpu_params)

    torch.cuda.empty_cache()

    if (step + 1) % CONFIG["log_every"] == 0:
        recent_reward = sum(step_rewards[-10:]) / len(step_rewards[-10:])
        recent_kl = sum(step_kls[-10:]) / len(step_kls[-10:])
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"Step {step+1:3d}/{CONFIG['num_steps']} | "
              f"Reward: {recent_reward:+.3f} | "
              f"KL: {recent_kl:.3f} | "
              f"VRAM: {vram:.2f}GB")

        if (step + 1) % 50 == 0:
            print(f"  Prompt:   {prompt_text}")
            print(f"  Response: {response_text[:150]}\n")


# ── 9. Save ──────────────────────────────────────────────────
actor.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"\n✅ PPO model v2 saved to: {CONFIG['output_dir']}")


# ── 10. Final Comparison ─────────────────────────────────────
print("\n" + "="*60)
print("FINAL COMPARISON — Base vs PPO v1 vs PPO v2")
print("="*60)

def generate(mdl, tok, prompt, max_new=60):
    inputs = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.3,
            use_cache=False,
            pad_token_id=tok.eos_token_id
        )
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

torch.cuda.empty_cache()

base_tok = AutoTokenizer.from_pretrained("gpt2")
base_tok.pad_token = base_tok.eos_token
base_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

v1_tok = AutoTokenizer.from_pretrained("/kaggle/input/datasets/arnavx/mini-instructgpt-models/ppo_model/ppo_model", local_files_only=True)
v1_tok.pad_token = v1_tok.eos_token
ppo_v1 = AutoModelForCausalLM.from_pretrained("/kaggle/input/datasets/arnavx/mini-instructgpt-models/ppo_model/ppo_model", local_files_only=True).to(device)

test_prompts = [
    "Human: What is the capital of France?\n\nAssistant:",
    "Human: How do I make a cup of tea?\n\nAssistant:",
    "Human: Can you explain what gravity is?\n\nAssistant:",
    "Human: What are some tips for studying better?\n\nAssistant:",
]

for prompt in test_prompts:
    print(f"\n📝 {prompt}")
    print(f"  🔴 Base GPT-2 : {generate(base_model, base_tok, prompt)[:150]}")
    print(f"  🟡 PPO v1     : {generate(ppo_v1, v1_tok, prompt)[:150]}")
    print(f"  🟢 PPO v2     : {generate(actor, tokenizer, prompt)[:150]}")

print("\n" + "="*60)
print("🎉 Mini InstructGPT v2 Complete!")
print("="*60)
print("\nWhat you built:")
print("  ✅ GPT-2 Medium SFT (345M, 20k samples)")
print("  ✅ Self-play data generation (2000 pairs)")
print("  ✅ Reward model v2 — 76.5% accuracy (was 63%)")
print("  ✅ PPO v2 with CPU-offloaded optimizer")
print("  ✅ Novel: self-play inspired by SPIN paper")

Writing /kaggle/working/phase4_ppo_v2.py


In [4]:
!wc -l /kaggle/working/phase4_ppo_v2.py

352 /kaggle/working/phase4_ppo_v2.py


In [5]:
!head -n 20 /kaggle/working/phase4_ppo_v2.py

# ============================================================
# Mini InstructGPT v2 — Phase 4: PPO Training
# Fix: optimizer states on CPU (saves ~1.4GB VRAM)
# GPT-2 Medium is too large for full GPU training on 4GB
# ============================================================

import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn.functional as F
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW

# ── 1. Config ────────────────────────────────────────────────
CONFIG = {
    "sft_model_path":    "/kaggle/input/datasets/arnavx/mini-instructgpt-models/sft_model_v2/sft_model_v2",


In [1]:
!python /kaggle/working/phase4_ppo_v2.py

python3: can't open file '/kaggle/working/phase4_ppo_v2.py': [Errno 2] No such file or directory


In [1]:
!ls -lah /kaggle/input/datasets/arnavx/mini-instructgpt-models

total 4.0K
drwxr-xr-x 5 nobody nogroup    0 Apr  3 08:02 .
drwxr-xr-x 3 root   root    4.0K Apr  3 08:03 ..
drwxr-xr-x 3 nobody nogroup    0 Apr  3 08:02 ppo_model
drwxr-xr-x 3 nobody nogroup    0 Apr  3 08:02 reward_model_v2
drwxr-xr-x 3 nobody nogroup    0 Apr  3 08:02 sft_model_v2


In [4]:
!python /kaggle/working/phase4_ppo_v2.py

Using device: cuda

Loading actor on GPU...
Loading weights: 100%|█| 292/292 [00:00<00:00, 306.85it/s, Materializing param=t
Actor loaded. VRAM: 1.44GB

Loading reference model on CPU...
Loading weights: 100%|█| 292/292 [00:00<00:00, 1580.04it/s, Materializing param=
Reference on CPU. VRAM: 1.44GB

Loading reward model v2 on CPU...
Loading weights: 100%|█| 293/293 [00:00<00:00, 637.10it/s, Materializing param=t
All models loaded. VRAM: 1.44GB

Loading safe PPO prompts...
README.md: 8.20kB [00:00, 16.3MB/s]
databricks-dolly-15k.jsonl: 100%|██████████| 13.1M/13.1M [00:00<00:00, 15.6MB/s]
Generating train split: 100%|██| 15011/15011 [00:00<00:00, 137962.79 examples/s]
Loaded 500 safe prompts

Starting PPO v2 — 120 steps
Actor GPU | Reference CPU | Optimizer CPU offloaded

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
Step  10/120 | Reward: +0.201 | KL: -0.045 | VRAM: 2.90GB
Step  20/120 | Reward: +0.378 | KL: -0.099 | VRAM: 2.90GB
Step  30/120 

In [5]:
!ls -lah /kaggle/working/ppo_model_v2

total 1.4G
drwxr-xr-x 2 root root 4.0K Apr  3 08:25 .
drwxr-xr-x 4 root root 4.0K Apr  3 08:25 ..
-rw-r--r-- 1 root root 1014 Apr  3 08:25 config.json
-rw-r--r-- 1 root root  118 Apr  3 08:25 generation_config.json
-rw-r--r-- 1 root root 1.4G Apr  3 08:25 model.safetensors
-rw-r--r-- 1 root root  295 Apr  3 08:25 tokenizer_config.json
-rw-r--r-- 1 root root 3.4M Apr  3 08:25 tokenizer.json


In [6]:
!zip -r /kaggle/working/ppo_model_v2.zip /kaggle/working/ppo_model_v2

  adding: kaggle/working/ppo_model_v2/ (stored 0%)
  adding: kaggle/working/ppo_model_v2/tokenizer_config.json (deflated 48%)
  adding: kaggle/working/ppo_model_v2/generation_config.json (deflated 25%)
  adding: kaggle/working/ppo_model_v2/tokenizer.json (deflated 82%)
  adding: kaggle/working/ppo_model_v2/config.json (deflated 54%)
  adding: kaggle/working/ppo_model_v2/model.safetensors (deflated 7%)


In [7]:
!ls -lh /kaggle/working/ppo_model_v2.zip

-rw-r--r-- 1 root root 1.3G Apr  3 08:48 /kaggle/working/ppo_model_v2.zip
